# Exploracion del dataset F1 2022

Este notebook carga `F1_2022_data.csv` en un DataFrame de pandas y responde:

- How big is it and what's in it?
- What type is each column, really?
- Is anything missing?
- Are there duplicates?
- What's the range of each number and does it make sense?

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('..') / 'data' / 'F1_2022_data.csv'
df = pd.read_csv(DATA_PATH)

print(f'Dataset cargado desde: {DATA_PATH.resolve()}')
df.head()

ModuleNotFoundError: No module named 'pandas'

## 1. Tamano y contenido

La forma del DataFrame indica el numero de filas y columnas. La tabla siguiente resume cada columna y muestra algunos valores representativos.

In [ ]:
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Tamano total: {df.shape[0] * df.shape[1]} celdas')

column_summary = pd.DataFrame({
    'column': df.columns,
    'pandas_dtype': df.dtypes.astype(str).values,
    'non_null': df.notna().sum().values,
    'unique_values': df.nunique(dropna=True).values,
    'example_values': [df[column].dropna().head(3).tolist() for column in df.columns],
})
column_summary

## 2. Tipo real de cada columna

Primero observamos los tipos inferidos por pandas. Despues usamos `convert_dtypes()` para obtener tipos pandas mas expresivos, como `string` e `Int64`, y comparamos ambos resultados.

In [ ]:
typed_df = df.convert_dtypes()
type_summary = pd.DataFrame({
    'column': df.columns,
    'inferred_by_read_csv': df.dtypes.astype(str).values,
    'pandas_nullable_type': typed_df.dtypes.astype(str).values,
})
type_summary

**Interpretacion:** `POS`, `Points`, `Pole Positions`, `No of Fastest Laps`, `Wins`, `Podiums` y `DNFs` son conteos enteros. `Driver Code` es un identificador de texto, mientras que `Driver Name` y `Constructor` son categorias de texto.

## 3. Valores faltantes

Contamos valores nulos y tambien cadenas vacias o con espacios, por si una celda parece vacia pero no fue interpretada como `NaN`.

In [ ]:
missing_summary = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2),
    'blank_strings': df.apply(
        lambda column: column.astype('string').str.strip().eq('').sum()
    ),
})
print(f'Valores faltantes totales: {int(df.isna().sum().sum())}')
print(f'Cadenas vacias totales: {int(missing_summary["blank_strings"].sum())}')
missing_summary

## 4. Duplicados

Buscamos filas completamente repetidas y tambien comprobamos si hay identificadores repetidos en `POS`, `Driver Code` o `Driver Name`.

In [ ]:
duplicate_rows = int(df.duplicated().sum())
print(f'Filas duplicadas completas: {duplicate_rows}')

key_columns = ['POS', 'Driver Code', 'Driver Name']
duplicate_keys = pd.DataFrame({
    'column': key_columns,
    'duplicated_values': [int(df[column].duplicated().sum()) for column in key_columns],
    'unique_values': [int(df[column].nunique()) for column in key_columns],
})
duplicate_keys

## 5. Rangos numericos y plausibilidad

La tabla muestra minimo, maximo, media y valores distintos de cada columna numerica. Las comprobaciones de dominio verifican que la posicion sea unica y consecutiva, que los conteos no sean negativos, que los podios no sean menores que las victorias y que las victorias no superen los podios.

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns
numeric_ranges = df[numeric_columns].agg(['min', 'max', 'mean', 'nunique']).T
numeric_ranges.columns = ['min', 'max', 'mean', 'unique_values']
numeric_ranges.round(2)

In [ ]:
checks = {
    'POS unica': df['POS'].is_unique,
    'POS consecutiva de 1 a n': df['POS'].sort_values().tolist() == list(range(1, len(df) + 1)),
    'Puntos no negativos': (df['Points'] >= 0).all(),
    'Pole Positions no negativas': (df['Pole Positions'] >= 0).all(),
    'Fastest Laps no negativas': (df['No of Fastest Laps'] >= 0).all(),
    'Victorias no negativas': (df['Wins'] >= 0).all(),
    'Podios no negativos': (df['Podiums'] >= 0).all(),
    'DNFs no negativos': (df['DNFs'] >= 0).all(),
    'Victorias <= podios': (df['Wins'] <= df['Podiums']).all(),
}
plausibility = pd.Series(checks, name='passes_check')
plausibility

if plausibility.all():
    print('Todas las comprobaciones de plausibilidad pasaron.')
else:
    print('Al menos una comprobacion requiere revision:')
    display(plausibility[~plausibility])

## Conclusion

El dataset contiene una fila por piloto de F1 2022 y columnas con su posicion, identificadores, constructor y resultados estadisticos. Las tablas anteriores dejan documentados su tamano, tipos, calidad de datos y rangos; la ultima salida indica si los valores numericos son coherentes con las reglas basicas del dominio.